# Milestone 5 lab, notebook 3 of 8: Five signals

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| **3. Five signals** (you are here) | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| 7. Four drills | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes.** The week from notebook 2 is used. If it is not there, Airflow runs it first: about four minutes.

In [1]:
import pandas as pd
from lab_helpers import (
    LAST_NIGHT, WEEK, airflow_runs, airflow_tasks, can_be_paid, prepare_nights, sql,
)
from lotus_pipeline import consumers

prepare_nights(LAST_NIGHT)

The bonus week ran under Airflow: 7 nights, Mon 08-24 to Sun 08-30. 1 of 6 stores can be paid.


## Step 1: Green every night

**Dana Okafor** runs store `S02`. On Friday 28 August her store's sales file was sent twice. Her Friday came in at **226%** of a normal Friday. Every Airflow task was green that night.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

Ask Airflow about the week:

In [2]:
runs = airflow_runs()
tasks = pd.concat([airflow_tasks(night) for night in WEEK])
print("Runs:", len(runs), " succeeded:", int((runs["state"] == "success").sum()))
print("Tasks:", len(tasks), " succeeded:", int((tasks["state"] == "success").sum()))
print("Stores that can be paid on Sunday night:", can_be_paid(LAST_NIGHT), "of 6")

Runs: 7  succeeded: 7
Tasks: 56  succeeded: 56
Stores that can be paid on Sunday night: 1 of 6


Seven runs, 56 tasks, all green. One store of six can be paid.

Watching whether jobs run is called **monitoring**. It is needed, and it is not enough. Lotus had monitoring before this project, and it was green while the numbers were wrong.

**Observability** is being able to tell, from outside, whether the **data** is right, not only whether the job ran.

| | Monitoring | Observability |
|---|---|---|
| It asks | Did the job run and finish? | Is the data right? |
| It reads | Airflow's run history | The data, and what is known about it |
| Lotus this week | 56 of 56 tasks green | 1 of 6 stores can be paid, and each reason is known |

## Step 2: Five signals

Observability watches five signals. Some books call them the **five pillars**. Each one already has a home in this project. You built every one of them; here they are side by side, each with one query on Lotus's rows.

| Signal | It asks | Built in |
|---|---|---|
| **Freshness** | Did the data arrive in time? | Milestone 5, notebook 2 |
| **Volume** | Is the amount of data normal? | Milestone 2: a store's day against its normal day |
| **Schema** | Did the columns change? | Milestone 3: the first line of every file is kept |
| **Distribution** | Do the values look normal? | Milestones 1 and 2: profiling, and a believable weight |
| **Lineage** | What does this table feed, and who reads it? | Milestone 1: the map of what reads what |

## Step 3: Freshness

Which sales files of the week came after 01:00?

In [3]:
sql("""
    SELECT file_name, received_at
    FROM raw.loaded_files
    WHERE table_name = 'sales_lines'
      AND received_at > CAST(substr(file_name, 17, 10) AS DATE) + 1 + INTERVAL '1 hour'
      AND substr(file_name, 17, 10) BETWEEN '2026-08-24' AND '2026-08-30'
""")

,file_name,received_at
0,sales_lines_S06_2026-08-27.csv,2026-08-29 00:40:00


One: Aiko's Thursday file, from notebook 2.

## Step 4: Volume

**Volume** is how much data arrived. You met it in milestone 2: a volume check asks whether a store's day has about as many lines as its normal day. Here the normal day is the average of the same weekday in the four weeks before. Inside 60% to 140% is normal.

The test kept every store day outside that range:

In [4]:
sql("SELECT * FROM audit.normal_volume ORDER BY sales_date, store_id")

,store_id,sales_date,lines,normal_lines,percent_of_normal
0,S02,2026-08-28,334,148,226


Dana's Friday: 226% of a normal Friday. Her file was sent twice, so every line came twice.

Now the opposite: a fault volume does **not** see. Tomas Reyes's refund at `S03` on Monday had a wrong sign. One line, wrong, in a file of normal size:

In [5]:
sql("SELECT count(*) AS s03_days_far_from_normal FROM audit.normal_volume WHERE store_id = 'S03'")

,s03_days_far_from_normal
0,0


Nothing. A volume check counts rows. It cannot see a wrong value in a row. And a day far from normal is only a reason to look: a public holiday is low, and it is not an error.

## Step 5: Schema

A **schema** is the list of columns and their types. The load step keeps the first line of every file: the columns as the sender wrote them. Here are the last two files from supplier `SUP-B`:

In [6]:
sql("""
    SELECT file_name, string_agg(column_name, ', ' ORDER BY position_in_file) AS columns_in_the_file
    FROM raw.file_columns
    WHERE file_name LIKE 'supplier_prices_SUP-B%'
    GROUP BY file_name
    ORDER BY file_name DESC
    LIMIT 2
""")

,file_name,columns_in_the_file
0,supplier_prices_SUP-B_2026-08-25.csv,"supplier_id, product_id, cost_per_unit, unit, price_date"
1,supplier_prices_SUP-B_2026-08-18.csv,"supplier_id, product_id, unit_cost, unit, price_date"


On 25 August the supplier renamed `unit_cost` to `cost_per_unit`. Every row is still a valid row. Only the list of columns shows it.

## Step 6: Distribution

**Distribution** asks whether the values look normal: the smallest, the largest, the usual. The five largest sales lines of the week:

In [7]:
sql("""
    SELECT store_id, CAST(sold_at AS DATE) AS sales_date, product_id, quantity,
           CAST(amount_cents AS INTEGER) AS amount_cents
    FROM raw.sales_lines
    WHERE CAST(sold_at AS DATE) BETWEEN DATE '2026-08-24' AND DATE '2026-08-30'
    ORDER BY CAST(amount_cents AS INTEGER) DESC
    LIMIT 5
""")

,store_id,sales_date,product_id,quantity,amount_cents
0,S05,2026-08-27,P-012,1180.000,152220
1,S02,2026-08-29,P-032,400.000,23600
2,S06,2026-08-26,P-022,3.000,3585
3,S03,2026-08-25,P-022,3.000,3585
4,S01,2026-08-25,P-022,3.000,3585


The first line is bananas (`P-012`) weighed in grams, not kilograms, at `S05` on Thursday: a quantity of 1,180 "kilograms" and an amount of $1,522.20. That is a fault.

The second line looks just as strange: 400 bottles of water (`P-032`) at `S02` on Saturday. It is a real sale: a café bought 400 bottles at $0.59. A large value is a reason to look, not a proof, like a large volume. A check that fired on the water would be switched off within a week.

## Step 7: Lineage

**Lineage** is the map of which table feeds which, and who reads the result. When a table is wrong, lineage says who is hurt. You drew it in milestone 1:

In [8]:
print(consumers.format_impact(consumers.impact("sales_lines")))

sales_lines: Each till writes it. Each store sends a file every night.
  A wrong value here reaches these tables:
    clean.sales_lines
    output.daily_store_sales
    output.replenishment_list
  and these people:
    Store managers
    Head of retail operations
    The replenishment team
    Finance


When `sales_lines` is wrong, four groups of people are hurt. That is why every alert in notebook 4 names one person.

Real teams use tools that draw lineage by themselves, from the SQL. This project does not install one; the map is written by hand in `lotus_pipeline/consumers.py`.

## What you now know

- **Monitoring** asks whether the job ran. **Observability** asks whether the data is right. Airflow was green 56 times out of 56, and one store of six could be paid.
- Observability watches five signals, also called the five pillars: **freshness**, **volume**, **schema**, **distribution** and **lineage**.
- **Volume** is how much data arrived. It saw a file sent twice, at 226%. It did not see a refund with the wrong sign.
- Each signal is a reason to look, not a proof. The checks of milestones 2 to 4 are where a reason becomes a decision.

**Say it in an interview:** "Monitoring told us the job was green every night. Observability asks whether the data is right. I watch five signals: freshness, volume, schema, distribution and lineage, and each one mapped to something we built. A store's file sent twice came in at 226% of a normal Friday. Volume is one signal, and only a reason to look."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), sections on the five pillars.

Next: [notebook 4](04_tell_the_person_who_can_act.ipynb). We know. Now tell somebody.